In [ ]:
suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(jsonlite)
})
BUCKET   <- Sys.getenv("WORKSPACE_BUCKET")
path_out <- file.path(BUCKET, "sjl/outputs")
read_gcs <- function(rel_path) {
  local <- paste0("/tmp/", basename(rel_path))
  system(paste("gsutil cp", file.path(path_out, rel_path), local), ignore.stdout = TRUE, ignore.stderr = TRUE)
  read_parquet(local)
}

fig2a_data    <- read_gcs("fig2a_sjl_distribution.parquet")
fig2b_data    <- read_gcs("fig2b_polar_midpoints.parquet")
fig2c_data    <- read_gcs("fig2c_weekly_sjl_heatmap.parquet")
logistic_scan <- read_gcs("phewas_logistic_primary.parquet")
cox_primary   <- read_gcs("cox_primary_results.parquet")
cox_sens_bmi  <- read_gcs("cox_sens_bmi_results.parquet")
cox_sens_apnea<- read_gcs("cox_sens_apnea_results.parquet")
pred_grid_raw <- read_gcs("cox_2d_pred_grid.parquet")
race_sjl      <- read_gcs("fig6a_sjl_by_race.parquet")
strat_race_raw<- read_gcs("cox_stratified_race.parquet")
cal_coverage  <- read_gcs("fig_calendar_coverage.parquet")
lm_vs_all     <- read_gcs("fig_landmark_vs_alltime_sjl.parquet")
weekly_long   <- read_gcs("supp_weekly_sjl_long.parquet")
logi_minimal  <- read_gcs("phewas_logistic_minimal.parquet")

inspect_data <- function(df, name) {
  cat("\n======================================================\n")
  cat("DATASET:", name, "| ROWS:", nrow(df), "| COLS:", ncol(df), "\n")
  cat("======================================================\n")
  cat("SCHEMA:\n")
  str(df)
  cat("\nFIRST 3 ROWS:\n")
  print(head(df, 3))
  cat("\n")
}

inspect_data(fig2a_data, "Figure 2A (Density)")
inspect_data(fig2b_data, "Figure 2B (Polar)")
inspect_data(fig2c_data, "Figure 2C (Heatmap)")
inspect_data(logistic_scan, "Figure 3 (Logistic PheWAS)")
inspect_data(cox_primary, "Figure 4 (Cox Forest)")
inspect_data(pred_grid_raw, "Figure 6 (2D Risk Surface)")
inspect_data(race_sjl, "Figure 7A (Race Density)")
inspect_data(strat_race_raw, "Figure 7B (Race Stratified Cox)")
inspect_data(cal_coverage, "Supp Fig 1 (Calendar Coverage)")
inspect_data(lm_vs_all, "Supp Fig 2A (Landmark vs All Time)")
inspect_data(weekly_long, "Supp Fig 2B (Weekly Long / Split Half)")

In [ ]:
library(tidyverse)
library(arrow)
library(viridis)
library(scales)
library(patchwork)

bucket_path <- Sys.getenv("WORKSPACE_BUCKET")
path_out <- paste0(bucket_path, "/sjl/outputs/")

nature_theme <- theme_minimal(base_size = 12) +
  theme(
    plot.tag = element_text(face = "bold", size = 16),
    plot.tag.position = "topleft",
    plot.margin = margin(t = 10, r = 20, b = 10, l = 10), 
    legend.title = element_text(face = "bold", size = 10),
    panel.grid.minor = element_blank()
  )


df_fig2a <- read_parquet(paste0(path_out, "fig2a_sjl_distribution.parquet"))

p33 <- quantile(df_fig2a$avg_sjl_hours, 0.333)
p67 <- quantile(df_fig2a$avg_sjl_hours, 0.667)

fig_density <- ggplot(df_fig2a, aes(x = avg_sjl_hours)) +
  geom_density(fill = "#3182bd", alpha = 0.85, color = NA) +
  geom_vline(xintercept = c(p33, p67), linetype = "dotted", color = "white", linewidth = 0.8) +
  annotate("label", x = p33 - 0.1, y = 0.1, label = sprintf("33rd\n%.2f", p33), 
           hjust = 1, color = "#1c5178", fontface = "bold", fill = "white", label.size = NA, alpha = 0.85) +
  annotate("label", x = p67 + 0.1, y = 0.1, label = sprintf("67th\n%.2f", p67), 
           hjust = 0, color = "#1c5178", fontface = "bold", fill = "white", label.size = NA, alpha = 0.85) +
  scale_x_continuous(breaks = 0:5) +
  coord_cartesian(xlim = c(0, 5)) +
  nature_theme +
  theme(
    panel.grid.major.y = element_blank(),
    panel.grid.major.x = element_line(color = "grey90"),
    axis.text.y = element_blank(),
    axis.title.y = element_blank()
  ) +
  labs(
    tag = "a",
    x = "Mean SJLsc (h)",
    y = NULL
  )


df_fig2b <- read_parquet(paste0(path_out, "fig2b_polar_midpoints.parquet")) %>%
  pivot_longer(cols = c(mean_msw_min, mean_msf_min), names_to = "day_type", values_to = "midpoint_min") %>%
  mutate(
    day_type = ifelse(day_type == "mean_msw_min", "Weekday", "Weekend"),
    midpoint_hour = midpoint_min / 60 
  )

fig_clock <- ggplot(df_fig2b, aes(x = midpoint_hour, fill = day_type, color = day_type)) +
  geom_density(alpha = 0.6, linewidth = 0.5) +
  coord_polar(theta = "x", start = 0, direction = 1) +
  scale_x_continuous(limits = c(0, 24), breaks = seq(0, 23, by = 3), labels = function(x) sprintf("%02d:00", x)) +
  scale_fill_manual(values = c("Weekday" = "#e6550d", "Weekend" = "#31a354")) +
  scale_color_manual(values = c("Weekday" = "#e6550d", "Weekend" = "#31a354")) +
  nature_theme +
  theme(
    axis.text.y = element_blank(),
    axis.title = element_blank(),
    panel.grid.major.x = element_line(color = "grey85", linewidth = 0.4),
    panel.grid.major.y = element_line(color = "grey95", linewidth = 0.4),
    legend.position = "top",
    legend.title = element_blank(),
    legend.margin = margin(b = -10)
  ) +
  labs(
    tag = "b",
    x = NULL,
    y = NULL
  )

df_fig2c <- read_parquet(paste0(path_out, "fig2c_weekly_sjl_heatmap.parquet"))

df_heatmap_base <- df_fig2c %>%
  group_by(person_id) %>%
  mutate(relative_week = as.integer(round(as.numeric(difftime(week_start, min(week_start), units = "weeks"))))) %>%
  ungroup()

ranks <- df_heatmap_base %>%
  group_by(person_id) %>%
  summarise(avg_sjl = first(avg_sjl_hours)) %>%
  mutate(sjl_percentile = ntile(avg_sjl, 100))

df_heatmap_smooth <- df_heatmap_base %>%
  inner_join(ranks, by = "person_id") %>%
  group_by(sjl_percentile, relative_week) %>%
  summarise(weekly_sjl_smooth = mean(sjl_sc_hours, na.rm = TRUE), .groups = "drop") %>%
  mutate(sjl_sc_hours_capped = ifelse(weekly_sjl_smooth > 4, 4, weekly_sjl_smooth))

fig_heatmap <- ggplot(df_heatmap_smooth, aes(x = relative_week, y = sjl_percentile, fill = sjl_sc_hours_capped)) +
  geom_raster(interpolate = FALSE) + 
  scale_fill_gradientn(
    colors = c("#1a9641", "#ffffbf", "#d7191c"),
    name = "Weekly SJLsc (h)", 
    limits = c(0, 4), 
    oob = squish,
    guide = guide_colorbar(barwidth = 1.2, barheight = 8, frame.colour = NA, ticks = FALSE)
  ) +
  scale_x_continuous(expand = c(0,0), breaks = seq(0, 26, by = 2)) +
  scale_y_continuous(expand = c(0,0), breaks = c(1, 50, 100), labels = c("Lowest", "Median", "Highest")) +
  nature_theme +
  theme(
    axis.ticks.y = element_blank(),
    panel.grid = element_blank(), 
    panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5), 
    legend.position = "right"
  ) +
  labs(
    tag = "c",
    x = "Observation week",
    y = "Cohort SJLsc percentile"
  )

df_cal <- read_parquet(paste0(path_out, "fig_calendar_coverage.parquet")) %>%
  mutate(
    week_idx = day_offset %/% 7,
    day_of_week = factor(weekday_num, levels = 7:1, labels = c("Sun", "Sat", "Fri", "Thu", "Wed", "Tue", "Mon"))
  )

fig_github <- ggplot(df_cal, aes(x = week_idx, y = day_of_week, fill = pct_coverage)) +
  geom_tile(color = "white", linewidth = 1.2) +
  scale_fill_gradient(
    low = "#ebedf0", high = "#196127", name = "Adherence", labels = percent,
    guide = guide_colorbar(barwidth = 1.2, barheight = 6, frame.colour = NA, ticks = FALSE)
  ) +
  scale_x_continuous(breaks = seq(0, 26, by = 2), expand = c(0,0)) +
  coord_fixed(ratio = 1, expand = FALSE) +
  nature_theme +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5), 
    axis.text.x = element_text(color = "black"),
    axis.ticks.x = element_line(color = "black"),
    legend.position = "right"
  ) +
  labs(
    tag = "d",
    x = "Observation week",
    y = "Day of week"
  )

final_layout <- (fig_density | fig_clock) / fig_heatmap / fig_github + 
  plot_layout(heights = c(1.5, 2, 0.8))

print(final_layout)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 6.5, repr.plot.res = 300)

library(tidyverse)
library(ggrepel)

total_tests <- 1500 
bonf_threshold <- -log10(0.05 / total_tests)

fdr_data <- logistic_scan %>% filter(fdr_q < 0.05)
fdr_threshold <- if(nrow(fdr_data) > 0) min(fdr_data$neg_log_p_value) else -log10(0.01)

padding_size <- 75 

plot_data <- logistic_scan %>%
  filter(!is.na(neg_log_p_value)) %>%
  arrange(phecode_category, phecode) %>%
  mutate(
    category_factor = as.numeric(as.factor(phecode_category)),
    pos = row_number() + (category_factor * padding_size)
  )

axis_df <- plot_data %>%
  group_by(phecode_category) %>%
  summarize(center = mean(pos), .groups = 'drop')

target_phecodes <- c() 

label_data <- plot_data %>%
  filter(phecode %in% target_phecodes)

p <- ggplot(plot_data, aes(x = pos, y = neg_log_p_value, color = as.factor(category_factor))) +
  geom_point(shape = 16, size = 1.5, alpha = 0.75) +
  scale_color_manual(values = rep(c("#3182bd", "#636363"), length.out = length(unique(plot_data$category_factor)))) +
  geom_text_repel(
    data = label_data,
    aes(label = phecode_string),
    size = 4,                 
    family = "sans",
    fontface = "bold",
    color = "black",          
    box.padding = 0.8,
    point.padding = 0.3,
    segment.color = "grey40",
    segment.size = 0.4,
    max.overlaps = Inf,       
    min.segment.length = 0,
    show.legend = FALSE
  ) +
  geom_hline(yintercept = bonf_threshold, linetype = "dashed", color = "#404040", linewidth = 0.5) +
  annotate("text", x = min(plot_data$pos), y = bonf_threshold + 0.3, label = "Bonferroni", color = "#404040", size = 3.5, hjust = 0, fontface = "italic") +
  geom_hline(yintercept = fdr_threshold, linetype = "dotted", color = "#404040", linewidth = 0.5) +
  annotate("text", x = min(plot_data$pos), y = fdr_threshold + 0.3, label = "FDR", color = "#404040", size = 3.5, hjust = 0, fontface = "italic") +
  scale_x_continuous(label = axis_df$phecode_category, breaks = axis_df$center) +
  scale_y_continuous(expand = expansion(mult = c(0.01, 0.15))) + 
  labs(
    x = NULL, 
    y = expression(paste(-log[10], "(", italic(p), ")"))
  ) +
  theme_classic(base_size = 14, base_family = "sans") +
  theme(
    text = element_text(color = "black"),
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black", size = 11),
    axis.text.y = element_text(color = "black", size = 11),
    axis.title.y = element_text(face = "bold", size = 13, margin = margin(r = 10)),
    axis.line = element_line(color = "black", linewidth = 0.5),
    axis.ticks = element_line(color = "black", linewidth = 0.5),
    axis.ticks.length = unit(0.15, "cm"),
    panel.grid = element_blank(),
    legend.position = "none", 
    plot.margin = margin(t = 20, r = 20, b = 10, l = 10)
  )

print(p)

In [ ]:
options(repr.plot.width = 6.5, repr.plot.height = 5, repr.plot.res = 300)

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(ggplot2)
})

BUCKET   <- Sys.getenv("WORKSPACE_BUCKET")
path_out <- file.path(BUCKET, "sjl/outputs")

read_gcs <- function(rel_path) {
  local <- paste0("/tmp/", basename(rel_path))
  system(paste("gsutil cp", file.path(path_out, rel_path), local), ignore.stdout = TRUE, ignore.stderr = TRUE)
  read_parquet(local)
}

lm_vs_all <- read_gcs("fig_landmark_vs_alltime_sjl.parquet")

ba_data <- lm_vs_all %>%
  filter(!is.na(avg_sjl_hours) & !is.na(alltime_avg_sjl_hours)) %>%
  mutate(
    mean_sjl = (avg_sjl_hours + alltime_avg_sjl_hours) / 2,
    diff_sjl = avg_sjl_hours - alltime_avg_sjl_hours
  )

bias  <- mean(ba_data$diff_sjl)
sd    <- sd(ba_data$diff_sjl)
upper <- bias + (1.96 * sd)
lower <- bias - (1.96 * sd)

x_max    <- 4.0
x_canvas <- 4.2
y_bound  <- 2.5

theme_stark <- theme_classic(base_size = 12, base_family = "sans") +
  theme(
    text = element_text(color = "black"),
    axis.title = element_text(face = "plain", size = 13, color = "black"), 
    axis.text = element_text(color = "#222222", size = 11), 
    axis.line = element_line(color = "black", linewidth = 0.3), 
    axis.ticks = element_line(color = "black", linewidth = 0.3),
    plot.title = element_blank(),
    plot.subtitle = element_blank(),
    plot.caption = element_blank(),
    panel.grid = element_blank(),
    legend.position = "none",
    plot.margin = margin(t = 20, r = 20, b = 15, l = 15) 
  )

ggplot(ba_data, aes(x = mean_sjl, y = diff_sjl)) +
  
  geom_point(color = "#1c2e3d", size = 0.6, alpha = 0.3, shape = 16, stroke = 0) +
  
  geom_hline(yintercept = 0, color = "black", linewidth = 0.4) + 
  geom_hline(yintercept = bias, color = "#555555", linetype = "longdash", linewidth = 0.3) + 
  geom_hline(yintercept = upper, color = "#555555", linetype = "dotted", linewidth = 0.3) + 
  geom_hline(yintercept = lower, color = "#555555", linetype = "dotted", linewidth = 0.3) + 
  
  annotate("text", x = x_max, y = bias, label = paste("Bias:", round(bias, 2)), 
           hjust = 1, vjust = -0.6, size = 3.5, fontface = "italic", color = "#444444") +
  annotate("text", x = x_max, y = upper, label = paste("+1.96 SD:", round(upper, 2)), 
           hjust = 1, vjust = -0.6, size = 3.5, fontface = "italic", color = "#444444") +
  annotate("text", x = x_max, y = lower, label = paste("-1.96 SD:", round(lower, 2)), 
           hjust = 1, vjust = 1.6, size = 3.5, fontface = "italic", color = "#444444") +
  
  scale_x_continuous(breaks = seq(0, 4, 1), expand = c(0, 0)) +
  scale_y_continuous(breaks = seq(-2, 2, 1), expand = c(0, 0)) +
  coord_cartesian(xlim = c(-0.05, x_canvas), ylim = c(-y_bound, y_bound)) +
  
  labs(
    x = "Mean SJLsc (h)",
    y = "Difference (h)"
  ) +
  theme_stark

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 11.5, repr.plot.res = 300)

suppressPackageStartupMessages({
  library(arrow)
  library(dplyr)
  library(stringr)
  library(ggplot2)
})

bucket_path <- Sys.getenv("WORKSPACE_BUCKET")
path_out    <- paste0(bucket_path, "/sjl/outputs/")

read_gcs <- function(rel_path) {
  local <- paste0("/tmp/", basename(rel_path))
  system(paste("gsutil cp", file.path(path_out, rel_path), local), ignore.stdout = TRUE, ignore.stderr = TRUE)
  read_parquet(local)
}

cox_primary    <- read_gcs("cox_primary_results.parquet")
cox_sens_bmi   <- read_gcs("cox_sens_bmi_results.parquet")
cox_sens_apnea <- read_gcs("cox_sens_apnea_results.parquet")


cox_p <- cox_primary %>% mutate(model = "Primary")
cox_b <- cox_sens_bmi %>% mutate(model = "+ BMI Adj.")
cox_a <- cox_sens_apnea %>% mutate(model = "+ Apnea Adj.")

cox_all <- bind_rows(cox_p, cox_b, cox_a) %>%
  filter(HR_hi < 10) %>% 
  mutate(nominal_sig = p_val < 0.05) %>%
  
  mutate(
    raw_clean = phecode_string %>%
      str_remove_all("\\[.*?\\]") %>%       
      str_remove_all("\\*$") %>%            
      str_trim()
  ) %>%
  
  mutate(
    clean_name = case_when(
      raw_clean == "Venous insufficiency (chronic) (peripheral)" ~ "Chronic venous insufficiency",
      raw_clean == "Elevated white blood cell count" ~ "Elevated WBC count",
      raw_clean == "Human immunodeficiency virus" ~ "HIV",
      raw_clean == "Tuberculosis test reaction" ~ "TB test reaction",
      raw_clean == "Nonspecific reaction to test for tuberculosis" ~ "TB test reaction",
      raw_clean == "Psychoactive substance dependance" ~ "Substance dependence", 
      raw_clean == "Current tobacco use and nicotine dependence" ~ "Tobacco & nicotine dependence",
      raw_clean == "Nicotine dependence (current and history of)" ~ "Nicotine dependence",
      raw_clean == "Psychoactive substance related disorders" ~ "Substance-related disorders",
      raw_clean == "Other symptoms and disorders of the respiratory system" ~ "Other respiratory disorders",
      raw_clean == "Dyspnea" ~ "Dyspnea",
      raw_clean == "Abnormalities of breathing" ~ "Breathing abnormalities",
      raw_clean == "Anxiety and anxiety disorders" ~ "Anxiety disorders",
      raw_clean == "Other disorders of liver" ~ "Other liver disorders",
      raw_clean == "Gastro-esophageal reflux disease" ~ "GERD",
      raw_clean == "Presence of cardiac device" ~ "Cardiac device presence",
      raw_clean == "Effusion of joint" ~ "Joint effusion",
      raw_clean == "Pain in leg" ~ "Leg pain",
      raw_clean == "Overweight and obesity" ~ "Overweight & obesity",
      TRUE ~ raw_clean
    )
  )

order_map <- cox_all %>%
  filter(model == "Primary") %>%
  arrange(HR) %>% 
  pull(clean_name) %>%
  unique()

cox_all <- cox_all %>%
  mutate(
    clean_name = factor(clean_name, levels = order_map),
    model = factor(model, levels = c("Primary", "+ BMI Adj.", "+ Apnea Adj."))
  )

nature_forest_theme <- theme_minimal(base_size = 12, base_family = "Helvetica") +
  theme(
    plot.margin = margin(t = 15, r = 25, b = 15, l = 10), 
    
    text = element_text(color = "#333333"),
    
    legend.position = "top",
    legend.title = element_text(face = "plain", size = 11, color = "#555555", margin = margin(r = 10)),
    legend.text = element_text(size = 11, color = "#333333"),
    legend.margin = margin(b = 20),
    
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(), 
    panel.grid.major.x = element_line(color = "grey92", linewidth = 0.3), # Very faint vertical guides
    
    axis.text.y = element_text(color = "#333333", size = 10, hjust = 1),
    axis.text.x = element_text(color = "#333333", size = 10),
    axis.line.x = element_line(color = "#333333", linewidth = 0.3), 
    axis.title.x = element_text(face = "plain", size = 12, color = "#333333", margin = margin(t = 15)),
    axis.title.y = element_blank(), 
    
    strip.text = element_text(face = "plain", size = 13, color = "#333333", margin = margin(b = 15)),
    panel.spacing = unit(2.5, "lines")
  )

model_colors <- c(
  "Primary"      = "#222222", 
  "+ BMI Adj."   = "#3182bd", 
  "+ Apnea Adj." = "#e6550d"  
)

ggplot(cox_all, aes(y = clean_name, color = model)) +
  
  geom_vline(xintercept = 1, color = "#555555", linewidth = 0.4) +
  
  geom_linerange(
    aes(xmin = HR_lo, xmax = HR_hi),
    linewidth = 0.45
  ) +
  
  geom_point(
    aes(x = HR, shape = nominal_sig),
    size = 2.4,    
    stroke = 0.7,  
    fill = "white" 
  ) +
  
  facet_wrap(~ model, nrow = 1) +
  
  scale_color_manual(values = model_colors, guide = "none") + 
  scale_shape_manual(
    values = c("TRUE" = 16, "FALSE" = 21), 
    name = "Significance",
    labels = c(expression(italic("p") >= 0.05), expression(italic("p") < 0.05))
  ) +
  
  scale_x_log10(
    breaks = c(0.8, 1.0, 1.25, 1.5, 2.0),
    labels = c("0.8", "1.0", "1.25", "1.5", "2.0")
  ) +
  
  coord_cartesian(xlim = c(0.75, 2.0)) +
  
  labs(
    x = "Hazard Ratio (95% CI)"
  ) +
  
  guides(
    shape = guide_legend(override.aes = list(color = "#333333", size = 3, stroke = 0.7))
  ) +
  
  nature_forest_theme

In [ ]:

library(jsonlite)
library(glue)
library(dplyr)
library(ggplot2)
library(ggconsort)

bucket <- Sys.getenv("WORKSPACE_BUCKET")
gcs_path <- paste0(bucket, "/sjl/outputs/consort_counts.json")
local_path <- "/tmp/consort_counts.json"
system2("gsutil", args = c("cp", gcs_path, local_path))

counts <- jsonlite::fromJSON(local_path)
fmt <- function(x) format(as.numeric(x), big.mark = ",")


lbl_main1 <- glue(
  "<b>All of Us Participants</b><br>
   <span style='font-size:10pt;'>(n = {fmt(counts$n_total)})</span>"
)

lbl_ex1 <- glue(
  "<b>Excluded (n = {fmt(counts$ex_sex)})</b><br>
   <span style='font-size:9pt;'>
   Sex at birth not recorded as male or female
   </span>"
)

lbl_main2 <- glue(
  "<b>Participants with Recorded Sex at Birth (Male or Female)</b><br>
   <span style='font-size:10pt;'>(n = {fmt(counts$n_sex)})</span>"
)

lbl_ex2 <- glue(
  "<b>Excluded (n = {fmt(counts$ex_sleep)})</b><br>
   <span style='font-size:9pt;'>
   No Fitbit main sleep episode data available
   </span>"
)

lbl_main3 <- glue(
  "<b>Participants with Fitbit-Derived Sleep Data</b><br>
   <span style='font-size:10pt;'>(n = {fmt(counts$n_sleep)})</span>"
)

lbl_ex3 <- glue(
  "<b>Excluded (n = {fmt(counts$ex_survey_ehr)})</b><br>
   <span style='font-size:9pt;'>
   Missing baseline survey covariates<br>or EHR follow-up data
   </span>"
)

lbl_main4 <- glue(
  "<b>Participants with Complete Survey and EHR Data</b><br>
   <span style='font-size:10pt;'>(n = {fmt(counts$n_survey_ehr)})</span>"
)

lbl_ex4 <- glue(
  "<b>Excluded (n = {fmt(counts$ex_sjl)})</b><br>
   <span style='font-size:9pt;'>
   Insufficient sleep observations within<br>182-day landmark window<br>
   (&lt;14 total nights or &lt;4 weekend nights)<br>
   or insufficient weekly distribution<br>
   (&lt;3 weekdays or &lt;1 weekend night per week)
   </span>"
)

lbl_main5 <- glue(
  "<b>Participants with Valid Sleep-Corrected Social Jetlag (SJL<sub>sc</sub>)</b><br>
   <span style='font-size:10pt;'>(n = {fmt(counts$n_sjl)})</span>"
)

lbl_ex5 <- glue(
  "<b>Excluded (n = {fmt(counts$ex_steps)})</b><br>
   <span style='font-size:9pt;'>
   Missing or implausible physical activity data<br>
   (median daily steps ≤ 500)
   </span>"
)

lbl_main6 <- glue(
  "<b>Participants with Valid Physical Activity Data</b><br>
   <span style='font-size:10pt;'>(n = {fmt(counts$n_steps)})</span>"
)

lbl_ex6 <- glue(
  "<b>Excluded (n = {fmt(counts$ex_ehr_len)})</b><br>
   <span style='font-size:9pt;'>
   Implausible EHR follow-up duration (&gt;40 years)
   </span>"
)

lbl_main7 <- glue(
  "<b>Final Analytic Cohort</b><br>
   <span style='font-size:10pt;'>
   Included in PheWAS and time-to-event analyses<br>
   (n = {fmt(counts$n_final)})
   </span>"
)

y_main <- seq(36, 0, by = -6)
y_ex   <- y_main - 2

dummy_cohort <- data.frame(id = 1) %>% cohort_start("Dummy")

sjl_consort <- dummy_cohort %>%

  consort_box_add("m1", 0,  y_main[1], lbl_main1) %>%
  consort_box_add("m2", 0,  y_main[2], lbl_main2) %>%
  consort_box_add("m3", 0,  y_main[3], lbl_main3) %>%
  consort_box_add("m4", 0,  y_main[4], lbl_main4) %>%
  consort_box_add("m5", 0,  y_main[5], lbl_main5) %>%
  consort_box_add("m6", 0,  y_main[6], lbl_main6) %>%
  consort_box_add("m7", 0,  y_main[7], lbl_main7) %>%

  consort_box_add("e1", 14, y_ex[1], lbl_ex1) %>%
  consort_box_add("e2", 14, y_ex[2], lbl_ex2) %>%
  consort_box_add("e3", 14, y_ex[3], lbl_ex3) %>%
  consort_box_add("e4", 14, y_ex[4], lbl_ex4) %>%
  consort_box_add("e5", 14, y_ex[5], lbl_ex5) %>%
  consort_box_add("e6", 14, y_ex[6], lbl_ex6) %>%

  consort_arrow_add("m1", "bottom", "m2", "top") %>%
  consort_arrow_add("m2", "bottom", "m3", "top") %>%
  consort_arrow_add("m3", "bottom", "m4", "top") %>%
  consort_arrow_add("m4", "bottom", "m5", "top") %>%
  consort_arrow_add("m5", "bottom", "m6", "top") %>%
  consort_arrow_add("m6", "bottom", "m7", "top") %>%

  consort_arrow_add(end="e1", end_side="left", start_x=0, start_y=y_ex[1]) %>%
  consort_arrow_add(end="e2", end_side="left", start_x=0, start_y=y_ex[2]) %>%
  consort_arrow_add(end="e3", end_side="left", start_x=0, start_y=y_ex[3]) %>%
  consort_arrow_add(end="e4", end_side="left", start_x=0, start_y=y_ex[4]) %>%
  consort_arrow_add(end="e5", end_side="left", start_x=0, start_y=y_ex[5]) %>%
  consort_arrow_add(end="e6", end_side="left", start_x=0, start_y=y_ex[6])

options(repr.plot.width = 10, repr.plot.height = 14)

final_plot <- sjl_consort %>%
  ggplot() +
  geom_consort() +
  theme_consort() +
  coord_cartesian(xlim = c(-12, 28)) + 
  theme(
    text = element_text(family = "sans", color = "black"),
    plot.background = element_rect(fill = "white", color = NA),
    panel.background = element_rect(fill = "white", color = NA),
    plot.margin = margin(20, 20, 20, 20)
  )

print(final_plot)